In [4]:
from google.colab import drive
drive.mount('/content/drive')
!mv glove.6B.zip /content/drive/MyDrive/

Mounted at /content/drive


Task 1: Data Preparation

In [50]:
import numpy as np
import pandas as pd
import re
import string
import random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('wordnet')
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import mean_squared_error, f1_score, hamming_loss, jaccard_score
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import warnings
warnings.filterwarnings("ignore")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

Using device: cpu


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
path = "/content/drive/MyDrive/movies.csv"

df = pd.read_csv(path)
allowed_columns = ['overview', 'tagline', 'keywords', 'genres', 'vote_average']
df = df[allowed_columns]
df.head()
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())
text_columns = ['overview', 'tagline', 'keywords']

for col in text_columns:
    df[col] = df[col].fillna("")

df['genres'] = df['genres'].fillna("")
df['vote_average'] = df['vote_average'].fillna(df['vote_average'].mean())

Dataset shape: (4803, 5)

Missing values:
overview          3
tagline         844
keywords        412
genres           28
vote_average      0
dtype: int64


In [10]:

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # lowercase
    text = text.lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize
    tokens = word_tokenize(text)

    # lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # join back
    return " ".join(tokens)

for col in text_columns:
    print(f"Preprocessing {col}...")
    df[col] = df[col].apply(preprocess_text)

Preprocessing overview...
Preprocessing tagline...
Preprocessing keywords...


In [51]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nPercentages:")
print("Train:", len(train_df)/len(df))
print("Validation:", len(val_df)/len(df))
print("Test:", len(test_df)/len(df))

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

Train size: 3362
Validation size: 720
Test size: 721

Percentages:
Train: 0.6999791796793671
Validation: 0.14990630855715179
Test: 0.15011451176348115


Task 2: GloVe Embedding Pipeline

*   glove.6B.100d.txt
*   Embedding dimension = 100



In [52]:
GLOVE_PATH = "/content/glove.6B.100d.txt"

embedding_dim = 100

embeddings_index = {}

with open(GLOVE_PATH, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print("Total GloVe vectors loaded:", len(embeddings_index))
print("Embedding dimension:", embedding_dim)

Total GloVe vectors loaded: 400000
Embedding dimension: 100


In [14]:
def build_vocab(text_series):

    vocab = Counter()

    for text in text_series:
        tokens = text.split()
        vocab.update(tokens)

    return vocab

# Example: using overview column
vocab = build_vocab(train_df['overview'])

print("Total unique dataset tokens:", len(vocab))
covered = 0

for word in vocab:
    if word in embeddings_index:
        covered += 1

coverage = covered / len(vocab) * 100

print("Words covered by GloVe:", covered)
print("Total words:", len(vocab))
print("Embedding coverage: {:.2f}%".format(coverage))

Total unique dataset tokens: 17148
Words covered by GloVe: 15324
Total words: 17148
Embedding coverage: 89.36%


In [15]:
tfidf = TfidfVectorizer()
tfidf.fit(train_df['overview'])
tfidf_vocab = tfidf.vocabulary_

print("TF-IDF vocab size:", len(tfidf_vocab))

def document_embedding(text, tfidf, embeddings_index, embedding_dim):

    words = text.split()

    tfidf_vector = tfidf.transform([text])

    doc_embedding = np.zeros(embedding_dim)

    weight_sum = 0

    for word in words:

        if word in tfidf.vocabulary_ and word in embeddings_index:

            word_index = tfidf.vocabulary_[word]

            weight = tfidf_vector[0, word_index]

            doc_embedding += weight * embeddings_index[word]

            weight_sum += weight

    if weight_sum != 0:
        doc_embedding /= weight_sum

    return doc_embedding

TF-IDF vocab size: 17074


In [16]:
def create_embedding_matrix(text_series, tfidf, embeddings_index, embedding_dim):

    embeddings = []
    for text in text_series:

        emb = document_embedding(
            text,
            tfidf,
            embeddings_index,
            embedding_dim
        )
        embeddings.append(emb)
    return np.array(embeddings)


X_train_overview = create_embedding_matrix(
    train_df['overview'],
    tfidf,
    embeddings_index,
    embedding_dim
)
X_val_overview = create_embedding_matrix(
    val_df['overview'],
    tfidf,
    embeddings_index,
    embedding_dim
)
X_test_overview = create_embedding_matrix(
    test_df['overview'],
    tfidf,
    embeddings_index,
    embedding_dim
)

print("Train embedding shape:", X_train_overview.shape)
print("Validation embedding shape:", X_val_overview.shape)
print("Test embedding shape:", X_test_overview.shape)

Train embedding shape: (3362, 100)
Validation embedding shape: (720, 100)
Test embedding shape: (721, 100)


In [17]:
y_train = train_df['vote_average'].values
y_val = val_df['vote_average'].values
y_test = test_df['vote_average'].values

Task 3: Rating Prediction(Regression)

In [21]:
class MovieDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32

train_loader_overview = DataLoader(
    MovieDataset(X_train_overview, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader_overview = DataLoader(
    MovieDataset(X_val_overview, y_val),
    batch_size=batch_size
)

test_loader_overview = DataLoader(
    MovieDataset(X_test_overview, y_test),
    batch_size=batch_size
)

train_loader_tagline = DataLoader(
    MovieDataset(X_train_tagline, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader_tagline = DataLoader(
    MovieDataset(X_val_tagline, y_val),
    batch_size=batch_size
)

test_loader_tagline = DataLoader(
    MovieDataset(X_test_tagline, y_test),
    batch_size=batch_size
)

mean_rating = np.mean(y_train)

baseline_predictions = np.full_like(y_test, mean_rating)

baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(baseline_mse)

print("BASELINE MODEL")
print("Mean rating:", mean_rating)
print("MSE:", baseline_mse)
print("RMSE:", baseline_rmse)

BASELINE MODEL
Mean rating: 6.090065437239739
MSE: 1.2825133470102974
RMSE: 1.1324810581242837


In [22]:
class RegressionModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.output = nn.Linear(64, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.output(x)

        return x

In [24]:
def train_model(model, train_loader, val_loader, epochs=20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float("inf")
    best_model = None

    for epoch in range(epochs):

        # TRAIN
        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = criterion(predictions, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # VALIDATION
        model.eval()
        val_loss = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)

                loss = criterion(predictions, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}")
        print("Train Loss:", train_loss)
        print("Val Loss:", val_loss)
        print()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

    model.load_state_dict(best_model)

    return model

In [23]:
def evaluate_model(model, test_loader):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            predictions.extend(outputs.cpu().numpy())
            actual.extend(y_batch.numpy())

    mse = mean_squared_error(actual, predictions)
    rmse = np.sqrt(mse)

    return mse, rmse

In [25]:
model_overview = RegressionModel(embedding_dim)

model_overview = train_model(
    model_overview,
    train_loader_overview,
    val_loader_overview
)

mse_overview, rmse_overview = evaluate_model(
    model_overview,
    test_loader_overview
)

print("OVERVIEW MODEL")
print("MSE:", mse_overview)
print("RMSE:", rmse_overview)

Epoch 1
Train Loss: 8.611697139605036
Val Loss: 2.250871593537538

Epoch 2
Train Loss: 2.502650834479422
Val Loss: 2.026520653911259

Epoch 3
Train Loss: 2.310541657906658
Val Loss: 1.873225274293319

Epoch 4
Train Loss: 2.2060983237230554
Val Loss: 1.9803215809490369

Epoch 5
Train Loss: 2.119689261013607
Val Loss: 1.8079981907554294

Epoch 6
Train Loss: 2.1109719602566845
Val Loss: 1.6946998912355173

Epoch 7
Train Loss: 1.9440116241293133
Val Loss: 1.6780472371889197

Epoch 8
Train Loss: 1.9605236340243861
Val Loss: 1.677231101886086

Epoch 9
Train Loss: 1.963778701593291
Val Loss: 1.70864235577376

Epoch 10
Train Loss: 1.8995088957390696
Val Loss: 1.644022620242575

Epoch 11
Train Loss: 1.9276752702470095
Val Loss: 2.0946957417156384

Epoch 12
Train Loss: 1.8300514771550331
Val Loss: 1.6120978749316672

Epoch 13
Train Loss: 1.8714380894067153
Val Loss: 1.6569697545922322

Epoch 14
Train Loss: 1.830889639145923
Val Loss: 1.6098603398903557

Epoch 15
Train Loss: 1.8378748674437684
Va

In [26]:
model_tagline = RegressionModel(embedding_dim)

model_tagline = train_model(
    model_tagline,
    train_loader_tagline,
    val_loader_tagline
)

mse_tagline, rmse_tagline = evaluate_model(
    model_tagline,
    test_loader_tagline
)

print("TAGLINE MODEL")
print("MSE:", mse_tagline)
print("RMSE:", rmse_tagline)

Epoch 1
Train Loss: 12.081389837107569
Val Loss: 3.381398620812789

Epoch 2
Train Loss: 3.0205974342688076
Val Loss: 1.6361189743746882

Epoch 3
Train Loss: 2.1043289956056848
Val Loss: 1.707361921020176

Epoch 4
Train Loss: 2.0621567790238364
Val Loss: 1.6683891985727393

Epoch 5
Train Loss: 2.1170931137957663
Val Loss: 1.6493315385735554

Epoch 6
Train Loss: 2.084180711012966
Val Loss: 1.6405792469563691

Epoch 7
Train Loss: 2.032466319372069
Val Loss: 1.6133888597073762

Epoch 8
Train Loss: 2.0606406027416013
Val Loss: 1.6113659221193064

Epoch 9
Train Loss: 2.022121345097164
Val Loss: 1.7532127059024314

Epoch 10
Train Loss: 2.0887013971805573
Val Loss: 1.6294332094814465

Epoch 11
Train Loss: 1.9922919689484362
Val Loss: 1.578995054182799

Epoch 12
Train Loss: 1.925565703297561
Val Loss: 1.6053317992583565

Epoch 13
Train Loss: 1.9509170747028206
Val Loss: 1.6491009852160579

Epoch 14
Train Loss: 1.9806764283270206
Val Loss: 1.5920441435730976

Epoch 15
Train Loss: 1.9342704387205

In [27]:
print("\nFINAL COMPARISON")

print("\nBaseline")
print("RMSE:", baseline_rmse)

print("\nOverview")
print("RMSE:", rmse_overview)

print("\nTagline")
print("RMSE:", rmse_tagline)


FINAL COMPARISON

Baseline
RMSE: 1.1324810581242837

Overview
RMSE: 1.2121109645506651

Tagline
RMSE: 1.1687633379324147


Task 4: Genre Prediction(Multi-Label Classification)

In [28]:
def process_genres(genre_series):

    return genre_series.apply(lambda x: x.split())

train_genres = process_genres(train_df['genres'])
val_genres = process_genres(val_df['genres'])
test_genres = process_genres(test_df['genres'])

In [37]:
mlb = MultiLabelBinarizer()

y_train_genre = mlb.fit_transform(train_genres)
y_val_genre = mlb.transform(val_genres)
y_test_genre = mlb.transform(test_genres)

num_classes = len(mlb.classes_)

print("Number of genres:", num_classes)
print("Genres:", mlb.classes_)

Number of genres: 22
Genres: ['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Fiction' 'Foreign' 'History' 'Horror' 'Movie' 'Music'
 'Mystery' 'Romance' 'Science' 'TV' 'Thriller' 'War' 'Western']


In [31]:
class GenreDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [32]:
batch_size = 32

train_loader_overview = DataLoader(
    GenreDataset(X_train_overview, y_train_genre),
    batch_size=batch_size,
    shuffle=True
)

val_loader_overview = DataLoader(
    GenreDataset(X_val_overview, y_val_genre),
    batch_size=batch_size
)

test_loader_overview = DataLoader(
    GenreDataset(X_test_overview, y_test_genre),
    batch_size=batch_size
)

train_loader_tagline = DataLoader(
    GenreDataset(X_train_tagline, y_train_genre),
    batch_size=batch_size,
    shuffle=True
)

val_loader_tagline = DataLoader(
    GenreDataset(X_val_tagline, y_val_genre),
    batch_size=batch_size
)

test_loader_tagline = DataLoader(
    GenreDataset(X_test_tagline, y_test_genre),
    batch_size=batch_size
)

In [33]:
class GenreClassifier(nn.Module):

    def __init__(self, input_dim, num_classes):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_classes)

        )

    def forward(self, x):
        return self.network(x)

In [34]:
def train_genre_model(model, train_loader, val_loader, epochs=20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float("inf")

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}")
        print("Train Loss:", train_loss)
        print("Val Loss:", val_loss)
        print()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

    model.load_state_dict(best_model)

    return model

In [35]:
def evaluate_genre_model(model, test_loader):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            predictions.extend(preds.cpu().numpy())
            actual.extend(y_batch.numpy())

    predictions = np.array(predictions)
    actual = np.array(actual)

    micro_f1 = f1_score(actual, predictions, average='micro')
    macro_f1 = f1_score(actual, predictions, average='macro')

    hamming = hamming_loss(actual, predictions)

    jaccard = jaccard_score(actual, predictions, average='micro')

    return micro_f1, macro_f1, hamming, jaccard

In [38]:
model_overview_genre = GenreClassifier(
    embedding_dim,
    num_classes
)

model_overview_genre = train_genre_model(
    model_overview_genre,
    train_loader_overview,
    val_loader_overview
)

micro_f1_overview, macro_f1_overview, hamming_overview, jaccard_overview = evaluate_genre_model(
    model_overview_genre,
    test_loader_overview
)

print("OVERVIEW RESULTS")
print("Micro-F1:", micro_f1_overview)
print("Macro-F1:", macro_f1_overview)
print("Hamming Loss:", hamming_overview)
print("Jaccard Score:", jaccard_overview)

Epoch 1
Train Loss: 0.3897336354514338
Val Loss: 0.3019951601391253

Epoch 2
Train Loss: 0.30717492258211354
Val Loss: 0.2941665124634038

Epoch 3
Train Loss: 0.29416421027678363
Val Loss: 0.28144846791806427

Epoch 4
Train Loss: 0.2797314818456488
Val Loss: 0.26522392358468927

Epoch 5
Train Loss: 0.2691053977271296
Val Loss: 0.2560490026422169

Epoch 6
Train Loss: 0.26077389126678685
Val Loss: 0.25087699941966846

Epoch 7
Train Loss: 0.2569080017647653
Val Loss: 0.24743984380493994

Epoch 8
Train Loss: 0.25220265253534857
Val Loss: 0.24212895916855853

Epoch 9
Train Loss: 0.2497701861386029
Val Loss: 0.2399259222590405

Epoch 10
Train Loss: 0.2449479174782645
Val Loss: 0.23817921527053998

Epoch 11
Train Loss: 0.24281041844273513
Val Loss: 0.2358678365531175

Epoch 12
Train Loss: 0.24022290419857456
Val Loss: 0.23446767291297083

Epoch 13
Train Loss: 0.23922453680128422
Val Loss: 0.23375583148520926

Epoch 14
Train Loss: 0.23857889479061342
Val Loss: 0.23252914198066876

Epoch 15
Tra

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [39]:
model_tagline_genre = GenreClassifier(
    embedding_dim,
    num_classes
)

model_tagline_genre = train_genre_model(
    model_tagline_genre,
    train_loader_tagline,
    val_loader_tagline
)

micro_f1_tagline, macro_f1_tagline, hamming_tagline, jaccard_tagline = evaluate_genre_model(
    model_tagline_genre,
    test_loader_tagline
)

print("TAGLINE RESULTS")
print("Micro-F1:", micro_f1_tagline)
print("Macro-F1:", macro_f1_tagline)
print("Hamming Loss:", hamming_tagline)
print("Jaccard Score:", jaccard_tagline)

Epoch 1
Train Loss: 0.421522374423045
Val Loss: 0.3115969520548116

Epoch 2
Train Loss: 0.3091751308935993
Val Loss: 0.29877134898434515

Epoch 3
Train Loss: 0.30329354641572487
Val Loss: 0.2947453519572382

Epoch 4
Train Loss: 0.29842418179197133
Val Loss: 0.29318913299104443

Epoch 5
Train Loss: 0.2948942739727362
Val Loss: 0.2898982530054839

Epoch 6
Train Loss: 0.29280747178028216
Val Loss: 0.28934474095054297

Epoch 7
Train Loss: 0.2890700181981303
Val Loss: 0.2876203293385713

Epoch 8
Train Loss: 0.2876872056216564
Val Loss: 0.285542083175286

Epoch 9
Train Loss: 0.2853910840063725
Val Loss: 0.284161913006202

Epoch 10
Train Loss: 0.282877007042462
Val Loss: 0.28403262470079504

Epoch 11
Train Loss: 0.2823059803472375
Val Loss: 0.2833801631046378

Epoch 12
Train Loss: 0.2814553159306634
Val Loss: 0.2830748208191084

Epoch 13
Train Loss: 0.2795865352985994
Val Loss: 0.2822484594324361

Epoch 14
Train Loss: 0.2793559066529544
Val Loss: 0.28195636039194855

Epoch 15
Train Loss: 0.27

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("\nFINAL COMPARISON")

print("\nOverview")
print("Micro-F1:", micro_f1_overview)
print("Macro-F1:", macro_f1_overview)

print("\nTagline")
print("Micro-F1:", micro_f1_tagline)
print("Macro-F1:", macro_f1_tagline)

Task 5: Frequent words per genre

In [43]:
def process_genres(genre_series):
    return genre_series.apply(lambda x: x.split())

train_genres = process_genres(train_df['genres'])

genre_text_map = defaultdict(list)

for text, genres in zip(train_df['overview'], train_genres):

    words = text.split()

    for genre in genres:

        genre_text_map[genre].extend(words)

In [44]:
genre_word_freq = {}

for genre, words in genre_text_map.items():

    word_counts = Counter(words)

    genre_word_freq[genre] = word_counts

top_words_per_genre = {}

for genre, counter in genre_word_freq.items():

    top_words = counter.most_common(10)

    top_words_per_genre[genre] = top_words

bottom_words_per_genre = {}

min_freq = 3

for genre, counter in genre_word_freq.items():

    filtered_words = [(word, freq) for word, freq in counter.items() if freq >= min_freq]

    sorted_words = sorted(filtered_words, key=lambda x: x[1])

    bottom_words_per_genre[genre] = sorted_words[:10]

In [45]:
top_words_table = []

for genre, words in top_words_per_genre.items():

    for word, freq in words:

        top_words_table.append({
            "Genre": genre,
            "Word": word,
            "Frequency": freq
        })

top_words_df = pd.DataFrame(top_words_table)

top_words_df.sort_values(["Genre", "Frequency"], ascending=[True, False])

bottom_words_table = []

for genre, words in bottom_words_per_genre.items():

    for word, freq in words:

        bottom_words_table.append({
            "Genre": genre,
            "Word": word,
            "Frequency": freq
        })

bottom_words_df = pd.DataFrame(bottom_words_table)

bottom_words_df.sort_values(["Genre", "Frequency"])

print("TOP WORDS PER GENRE")
display(top_words_df)

print("\nBOTTOM WORDS PER GENRE")
display(bottom_words_df)

TOP WORDS PER GENRE


,Genre,Word,Frequency
0,Mystery,the,661
1,Mystery,a,647
2,Mystery,of,353
3,Mystery,to,352
4,Mystery,and,335
...,...,...,...
215,Movie,his,10
216,Movie,their,9
217,Movie,he,8
218,Movie,is,7



BOTTOM WORDS PER GENRE


,Genre,Word,Frequency
0,Mystery,awakens,3
1,Mystery,next,3
2,Mystery,link,3
3,Mystery,teenager,3
4,Mystery,rock,3
...,...,...,...
215,Movie,town,3
216,Movie,family,3
217,Movie,up,3
218,Movie,from,3


Task 6: Genre-Indicative Words Using TF-IDF

In [47]:
def process_genres(series):
    return series.apply(lambda x: x.split())

train_genres = process_genres(train_df['genres'])
test_genres = process_genres(test_df['genres'])

mlb = MultiLabelBinarizer()

y_train_genre = mlb.fit_transform(train_genres)
y_test_genre = mlb.transform(test_genres)

genre_names = mlb.classes_

print("Genres:", genre_names)

tfidf = TfidfVectorizer(
    max_features=10000,
    min_df=3,
    max_df=0.8
)

X_train_tfidf = tfidf.fit_transform(train_df['overview'])
X_test_tfidf = tfidf.transform(test_df['overview'])

feature_names = tfidf.get_feature_names_out()

print("TF-IDF shape:", X_train_tfidf.shape)

Genres: ['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Fiction' 'Foreign' 'History' 'Horror' 'Movie' 'Music'
 'Mystery' 'Romance' 'Science' 'TV' 'Thriller' 'War' 'Western']
TF-IDF shape: (3362, 5569)


In [ ]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver='liblinear'
    )
)

model.fit(X_train_tfidf, y_train_genre)

In [49]:
top_words_per_genre = {}

for i, genre in enumerate(genre_names):

    coefficients = model.estimators_[i].coef_[0]

    top_indices = np.argsort(coefficients)[-10:]

    top_words = [feature_names[j] for j in top_indices[::-1]]

    top_words_per_genre[genre] = top_words

indicative_words_table = []

for genre, words in top_words_per_genre.items():

    for rank, word in enumerate(words, 1):

        indicative_words_table.append({
            "Genre": genre,
            "Rank": rank,
            "Indicative Word": word
        })

indicative_words_df = pd.DataFrame(indicative_words_table)

indicative_words_df.sort_values(["Genre", "Rank"])

print("GENRE-INDICATIVE WORDS")
display(indicative_words_df)

GENRE-INDICATIVE WORDS


,Genre,Rank,Indicative Word
0,Action,1,agent
1,Action,2,cop
2,Action,3,ruthless
3,Action,4,criminal
4,Action,5,assassin
...,...,...,...
215,Western,6,sheriff
216,Western,7,lawman
217,Western,8,bounty
218,Western,9,bandit
